In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import random
import shutil
import numpy as np
import pandas as pd
import torch

PROJECT_DIR = "/content/drive/MyDrive/Tndrr_Task1/ task1_intent2json".replace(
    "/ task1", "/task1"
)

os.makedirs(PROJECT_DIR, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")
else:
    raise RuntimeError("Please enable a GPU in Colab Runtime settings.")

Mounted at /content/drive
Project directory: /content/drive/MyDrive/Tndrr_Task1/task1_intent2json
GPU available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [ ]:
!pip install -q transformers==4.46.3 peft==0.13.2 accelerate==1.1.1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
import torch
import transformers
import peft

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("CUDA:", torch.cuda.is_available())

PyTorch: 2.14.1+cu130
Transformers: 4.46.3
PEFT: 0.13.2
CUDA: True


In [ ]:
import random
import json
import os
from sklearn.model_selection import train_test_split
from collections import Counter

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

FIELDS = [
    "intent",
    "destination",
    "date",
    "time",
    "party_size",
    "item"
]

cities = [
    "Delhi", "Mumbai", "Hyderabad", "Bengaluru",
    "Chennai", "Pune", "Kolkata", "Jaipur",
    "Vijayawada", "Goa"
]

dates = [
    "tomorrow", "today", "December 5",
    "November 30", "next Monday",
    "December 12", "this weekend",
    "January 10"
]

times = [
    "7 PM", "6 PM", "8 AM", "9:30 AM",
    "12 PM", "5:30 PM", "10 AM"
]

cuisines = [
    "Indian", "Chinese", "Italian",
    "Mexican", "Japanese", "Thai"
]

products = [
    "Bluetooth speakers", "laptops",
    "wireless headphones", "smartphones",
    "running shoes", "smart watches"
]

tracking_ids = [
    f"ORD{random.randint(10000, 99999)}"
    for _ in range(200)
]

examples = []

def add_example(request, output):
    examples.append({
        "request": request,
        "output": output
    })

# 85 examples for each of six intents
for intent in [
    "restaurant_booking",
    "flight_search",
    "hotel_booking",
    "product_search",
    "weather_query",
    "order_tracking"
]:

    for i in range(85):

        city = random.choice(cities)
        date = random.choice(dates)
        time = random.choice(times)
        people = random.randint(1, 6)
        cuisine = random.choice(cuisines)

        if intent == "restaurant_booking":

            templates = [
                f"Book a table for {people} people in {city} on {date} at {time}.",
                f"Can you reserve a table for {people} people {date} at {time}?",
                f"I need a restaurant reservation in {city} for {people} guests.",
                f"Find a {cuisine} restaurant for {people} people on {date}.",
                f"Reserve a table for {people} at a {cuisine} restaurant at {time}."
            ]

            include_cuisine = random.choice([True, False])

            if include_cuisine:
                request = (
                    f"Find a {cuisine} restaurant in {city} "
                    f"for {people} people on {date} at {time}."
                )
                item = cuisine + " restaurant"
            else:
                request = (
                    f"Can you reserve a table for {people} people "
                    f"in {city} {date} at {time}?"
                )
                item = None

            output = {
                "intent": intent,
                "destination": city,
                "date": date,
                "time": time,
                "party_size": people,
                "item": item
            }

        elif intent == "flight_search":

            templates = [
                f"Find flights to {city} on {date}.",
                f"I want to fly to {city} {date}.",
                f"Search for an air ticket to {city} for {date}.",
                f"Show me available flights to {city}.",
                f"Book a flight to {city} departing {date}."
            ]

            request = random.choice(templates)

            output = {
                "intent": intent,
                "destination": city,
                "date": date if date.lower() in request.lower() else None,
                "time": None,
                "party_size": None,
                "item": None
            }

        elif intent == "hotel_booking":

            templates = [
                f"Find a hotel in {city} for {people} guests starting {date}.",
                f"Book a hotel room in {city} from {date} for {people} people.",
                f"I need accommodation in {city} for {people} guests.",
                f"Search for hotels in {city} starting {date}.",
                f"Reserve a hotel in {city} for {people} people."
            ]

            request = random.choice(templates)

            output = {
                "intent": intent,
                "destination": city,
                "date": date if date.lower() in request.lower() else None,
                "time": None,
                "party_size": (
                    people
                    if f"{people} guests" in request or f"{people} people" in request
                    else None
                ),
                "item": None
            }

        elif intent == "product_search":

            product = random.choice(products)

            templates = [
                f"Where can I buy {product}?",
                f"Find me {product}.",
                f"Search for the best {product}.",
                f"I want to purchase {product}.",
                f"Show me available {product}."
            ]

            request = random.choice(templates)

            output = {
                "intent": intent,
                "destination": None,
                "date": None,
                "time": None,
                "party_size": None,
                "item": product
            }

        elif intent == "weather_query":

            templates = [
                f"What is the weather in {city}?",
                f"Tell me the weather forecast for {city}.",
                f"Will it rain in {city} {date}?",
                f"Check today's weather in {city}.",
                f"How is the weather looking in {city}?"
            ]

            request = random.choice(templates)

            output = {
                "intent": intent,
                "destination": city,
                "date": date if date.lower() in request.lower() else None,
                "time": None,
                "party_size": None,
                "item": None
            }

        else:

            tracking_id = tracking_ids[i]

            templates = [
                f"Track my order {tracking_id}.",
                f"Where is shipment {tracking_id}?",
                f"Can you check the status of {tracking_id}?",
                f"I want to track parcel {tracking_id}.",
                f"Check delivery status for order {tracking_id}."
            ]

            request = random.choice(templates)

            output = {
                "intent": intent,
                "destination": None,
                "date": None,
                "time": None,
                "party_size": None,
                "item": tracking_id
            }

        add_example(request, output)

# Shuffle dataset
random.shuffle(examples)

# Stratified split: 400 / 50 / 60
labels = [x["output"]["intent"] for x in examples]

train_data, temp_data = train_test_split(
    examples,
    test_size=110,
    random_state=SEED,
    stratify=labels
)

temp_labels = [x["output"]["intent"] for x in temp_data]

val_data, test_data = train_test_split(
    temp_data,
    test_size=60,
    random_state=SEED,
    stratify=temp_labels
)

print("Total:", len(examples))
print("Training:", len(train_data))
print("Validation:", len(val_data))
print("Testing:", len(test_data))

print("\nIntent distribution:")
print(Counter(x["output"]["intent"] for x in examples))

# Save datasets directly to Drive
for filename, data in [
    ("train.jsonl", train_data),
    ("validation.jsonl", val_data),
    ("test.jsonl", test_data)
]:
    with open(os.path.join(PROJECT_DIR, filename), "w") as f:
        for row in data:
            f.write(json.dumps(row) + "\n")

print("\nDataset files saved to Google Drive.")

Total: 510
Training: 400
Validation: 50
Testing: 60

Intent distribution:
Counter({'order_tracking': 85, 'weather_query': 85, 'product_search': 85, 'flight_search': 85, 'hotel_booking': 85, 'restaurant_booking': 85})

Dataset files saved to Google Drive.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16
).to("cuda")

base_model.eval()

print("Model loaded:", MODEL_NAME)
print("Device:", next(base_model.parameters()).device)
print("Total parameters:",
      sum(p.numel() for p in base_model.parameters()))

Model loaded: Qwen/Qwen2.5-0.5B-Instruct
Device: cuda:0
Total parameters: 494032768


In [ ]:
import re

SYSTEM_PROMPT = """
You are an intent-to-JSON extraction system.

Convert the user's request into exactly one valid JSON object.

Use these six keys:
intent, destination, date, time, party_size, item

Allowed intents:
restaurant_booking, flight_search, hotel_booking,
product_search, weather_query, order_tracking

Rules:
- Use null for fields that are not applicable.
- party_size must be an integer or null.
- Do not invent information.
- Preserve the user's date wording.
- Return JSON only, without explanations or markdown.
"""

def generate_prediction(model, request):

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": request}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()


def extract_json(text):

    try:
        return json.loads(text)
    except:
        match = re.search(r"\{.*\}", text, re.DOTALL)

        if match:
            try:
                return json.loads(match.group())
            except:
                return None

    return None


def evaluate_model(model, test_data, description):

    results = []

    for example in test_data:

        raw = generate_prediction(
            model,
            example["request"]
        )

        prediction = extract_json(raw)

        results.append({
            "request": example["request"],
            "expected": example["output"],
            "predicted": prediction,
            "raw_response": raw
        })

    print(description, "evaluation completed:", len(results))

    return results


def calculate_metrics(results):

    total = len(results)

    valid = 0
    exact = 0
    intent_correct = 0
    field_correct = 0

    for row in results:

        pred = row["predicted"]
        expected = row["expected"]

        if isinstance(pred, dict):

            valid += 1

            if (
                set(pred.keys()) == set(FIELDS)
                and all(pred.get(k) == expected.get(k) for k in FIELDS)
            ):
                exact += 1

            if pred.get("intent") == expected["intent"]:
                intent_correct += 1

            for field in FIELDS:
                if pred.get(field) == expected[field]:
                    field_correct += 1

    return {
        "json_validity": round(valid / total * 100, 2),
        "exact_match": round(exact / total * 100, 2),
        "intent_accuracy": round(intent_correct / total * 100, 2),
        "field_accuracy": round(
            field_correct / (total * len(FIELDS)) * 100, 2
        )
    }

In [ ]:
baseline_results = evaluate_model(
    base_model,
    test_data,
    "Base model"
)

baseline_metrics = calculate_metrics(baseline_results)

print("\n===== BASE MODEL RESULTS =====")

for metric, value in baseline_metrics.items():
    print(f"{metric}: {value:.2f}%")

with open(
    os.path.join(PROJECT_DIR, "baseline_results.json"), "w"
) as f:
    json.dump(baseline_results, f, indent=2)

with open(
    os.path.join(PROJECT_DIR, "baseline_metrics.json"), "w"
) as f:
    json.dump(baseline_metrics, f, indent=2)

print("\nBase model results safely saved to Drive.")

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


Base model evaluation completed: 60

===== BASE MODEL RESULTS =====
json_validity: 100.00%
exact_match: 8.33%
intent_accuracy: 76.67%
field_accuracy: 52.78%

Base model results safely saved to Drive.


In [ ]:
from torch.utils.data import Dataset, DataLoader

class IntentDataset(Dataset):

    def __init__(self, examples):

        self.data = []

        for example in examples:

            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": example["request"]}
            ]

            prompt_text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            full_messages = messages + [{
                "role": "assistant",
                "content": json.dumps(
                    example["output"],
                    ensure_ascii=False
                )
            }]

            full_text = tokenizer.apply_chat_template(
                full_messages,
                tokenize=False,
                add_generation_prompt=False
            )

            prompt_ids = tokenizer(
                prompt_text,
                add_special_tokens=False
            )["input_ids"]

            full_ids = tokenizer(
                full_text,
                add_special_tokens=False,
                truncation=True,
                max_length=256
            )["input_ids"]

            prompt_length = len(prompt_ids)

            if prompt_length >= len(full_ids):
                raise ValueError("Invalid prompt/completion lengths")

            labels = full_ids.copy()
            labels[:prompt_length] = [-100] * prompt_length

            self.data.append({
                "input_ids": full_ids,
                "labels": labels
            })

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return self.data[index]


train_dataset = IntentDataset(train_data)
val_dataset = IntentDataset(val_data)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

Training samples: 400
Validation samples: 50


In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

base_model.config.use_cache = False

model = get_peft_model(
    base_model,
    lora_config
)

model.print_trainable_parameters()

trainable params: 4,399,104 || all params: 498,431,872 || trainable%: 0.8826


In [ ]:
from tqdm.auto import tqdm

BATCH_SIZE = 4
GRAD_ACCUMULATION = 4
EPOCHS = 3
LEARNING_RATE = 2e-4

def collate_fn(batch):

    max_len = max(len(x["input_ids"]) for x in batch)

    input_ids = []
    labels = []
    attention_masks = []

    for item in batch:

        padding = max_len - len(item["input_ids"])

        input_ids.append(
            item["input_ids"] +
            [tokenizer.pad_token_id] * padding
        )

        labels.append(
            item["labels"] + [-100] * padding
        )

        attention_masks.append(
            [1] * len(item["input_ids"]) +
            [0] * padding
        )

    return {
        "input_ids": torch.tensor(input_ids),
        "labels": torch.tensor(labels),
        "attention_mask": torch.tensor(attention_masks)
    }


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE
)


def evaluate_loss(loader):

    model.eval()

    total_loss = 0
    batches = 0

    with torch.no_grad():

        for batch in loader:

            batch = {
                k: v.to("cuda")
                for k, v in batch.items()
            }

            outputs = model(**batch)

            total_loss += outputs.loss.item()
            batches += 1

    model.train()

    return total_loss / batches


training_history = []

for epoch in range(EPOCHS):

    model.train()
    optimizer.zero_grad()

    total_loss = 0

    progress = tqdm(
        enumerate(train_loader),
        total=len(train_loader),
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for step, batch in progress:

        batch = {
            k: v.to("cuda")
            for k, v in batch.items()
        }

        outputs = model(**batch)
        loss = outputs.loss

        if not torch.isfinite(loss):
            raise ValueError("Non-finite loss detected")

        (loss / GRAD_ACCUMULATION).backward()

        if (
            (step + 1) % GRAD_ACCUMULATION == 0
            or (step + 1) == len(train_loader)
        ):

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )

            optimizer.step()
            optimizer.zero_grad()

        total_loss += loss.item()

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    train_loss = total_loss / len(train_loader)
    val_loss = evaluate_loss(val_loader)

    training_history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "validation_loss": val_loss
    })

    print(
        f"Epoch {epoch + 1}: "
        f"Train Loss={train_loss:.4f}, "
        f"Validation Loss={val_loss:.4f}"
    )

    # Save training history after every epoch
    with open(
        os.path.join(PROJECT_DIR, "training_history.json"), "w"
    ) as f:
        json.dump(training_history, f, indent=2)

print("\nTraining completed!")

Epoch 1/3:   0%|          | 0/100 [00:00<?, ?it/s]

Epoch 1: Train Loss=0.0160, Validation Loss=0.0023


Epoch 2/3:   0%|          | 0/100 [00:00<?, ?it/s]

Epoch 2: Train Loss=0.0009, Validation Loss=0.0017


Epoch 3/3:   0%|          | 0/100 [00:00<?, ?it/s]

Epoch 3: Train Loss=0.0011, Validation Loss=0.0037

Training completed!


In [ ]:
ADAPTER_DIR = os.path.join(
    PROJECT_DIR,
    "intent2json_lora_v2"
)

os.makedirs(ADAPTER_DIR, exist_ok=True)

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print("Adapter saved successfully!")
print("Adapter files:", os.listdir(ADAPTER_DIR))

print("\nTraining history:")
print(training_history)

Adapter saved successfully!
Adapter files: ['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer_config.json', 'special_tokens_map.json', 'added_tokens.json', 'vocab.json', 'merges.txt', 'tokenizer.json']

Training history:
[{'epoch': 1, 'train_loss': 0.016035941238369558, 'validation_loss': 0.002261194895254448}, {'epoch': 2, 'train_loss': 0.0009126147333881817, 'validation_loss': 0.0016849395984676308}, {'epoch': 3, 'train_loss': 0.0011100932750559877, 'validation_loss': 0.0036574219131860165}]


In [ ]:
model.config.use_cache = True
model.eval()

finetuned_results = evaluate_model(
    model,
    test_data,
    "Fine-tuned model"
)

finetuned_metrics = calculate_metrics(
    finetuned_results
)

print("\n===== FINE-TUNED MODEL RESULTS =====")

for metric, value in finetuned_metrics.items():
    print(f"{metric}: {value:.2f}%")

with open(
    os.path.join(PROJECT_DIR, "finetuned_results.json"), "w"
) as f:
    json.dump(finetuned_results, f, indent=2)

with open(
    os.path.join(PROJECT_DIR, "finetuned_metrics.json"), "w"
) as f:
    json.dump(finetuned_metrics, f, indent=2)

print("\nFine-tuned results saved to Drive.")

Fine-tuned model evaluation completed: 60

===== FINE-TUNED MODEL RESULTS =====
json_validity: 100.00%
exact_match: 100.00%
intent_accuracy: 100.00%
field_accuracy: 100.00%

Fine-tuned results saved to Drive.


In [ ]:
comparison = pd.DataFrame({
    "Metric": [
        "JSON validity",
        "Exact match accuracy",
        "Intent accuracy",
        "Field-level accuracy"
    ],

    "Base model (%)": [
        baseline_metrics["json_validity"],
        baseline_metrics["exact_match"],
        baseline_metrics["intent_accuracy"],
        baseline_metrics["field_accuracy"]
    ],

    "Fine-tuned model (%)": [
        finetuned_metrics["json_validity"],
        finetuned_metrics["exact_match"],
        finetuned_metrics["intent_accuracy"],
        finetuned_metrics["field_accuracy"]
    ]
})

comparison["Improvement (percentage points)"] = (
    comparison["Fine-tuned model (%)"] -
    comparison["Base model (%)"]
).round(2)

print(comparison.to_string(index=False))

comparison.to_csv(
    os.path.join(PROJECT_DIR, "model_comparison.csv"),
    index=False
)

              Metric  Base model (%)  Fine-tuned model (%)  Improvement (percentage points)
       JSON validity          100.00                 100.0                             0.00
Exact match accuracy            8.33                 100.0                            91.67
     Intent accuracy           76.67                 100.0                            23.33
Field-level accuracy           52.78                 100.0                            47.22


In [ ]:
failures = [
    row for row in finetuned_results
    if row["predicted"] != row["expected"]
]

print("Total fine-tuned model failures:", len(failures))

for i, row in enumerate(failures[:3], 1):

    print(f"\n===== FAILURE {i} =====")

    print("Request:", row["request"])

    print(
        "Expected:",
        json.dumps(row["expected"], indent=2)
    )

    print(
        "Predicted:",
        json.dumps(row["predicted"], indent=2)
    )

    print("Raw response:", row["raw_response"])

Total fine-tuned model failures: 0


In [ ]:
parameters = 7_000_000_000

# FP16 weights
fp16_weights = parameters * 2 / 1e9

# Full fine-tuning:
# FP16 weights (2 bytes)
# FP16 gradients (2 bytes)
# FP32 master weights (4 bytes)
# Adam optimizer states (8 bytes)
full_ft = parameters * 16 / 1e9

# LoRA:
# Frozen FP16 base weights plus adapters, optimizer and activations
lora_low = 16
lora_high = 24

# QLoRA:
# Quantized 4-bit base weights plus adapters, optimizer and activations
qlora_low = 6
qlora_high = 10

print("7B MODEL MEMORY ESTIMATES")
print(f"FP16 weights: {fp16_weights:.1f} GB")
print(f"Full fine-tuning: {full_ft:.1f} GB + activations")
print(f"LoRA: {lora_low}-{lora_high} GB (approx.)")
print(f"QLoRA: {qlora_low}-{qlora_high} GB (approx.)")

7B MODEL MEMORY ESTIMATES
FP16 weights: 14.0 GB
Full fine-tuning: 112.0 GB + activations
LoRA: 16-24 GB (approx.)
QLoRA: 6-10 GB (approx.)


In [ ]:
experiment_summary = {
    "model": MODEL_NAME,
    "dataset": {
        "total": len(examples),
        "training": len(train_data),
        "validation": len(val_data),
        "testing": len(test_data),
        "seed": SEED
    },
    "lora_settings": {
        "rank": 8,
        "alpha": 16,
        "dropout": 0.05,
        "target_modules": [
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj"
        ],
        "learning_rate": LEARNING_RATE,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "gradient_accumulation": GRAD_ACCUMULATION
    },
    "baseline_metrics": baseline_metrics,
    "finetuned_metrics": finetuned_metrics,
    "training_history": training_history,
    "failure_count": len(failures)
}

with open(
    os.path.join(PROJECT_DIR, "experiment_summary.json"), "w"
) as f:
    json.dump(experiment_summary, f, indent=2)

print("Experiment summary saved!")

Experiment summary saved!


In [ ]:
print("===== FINAL TASK 1 FILE CHECK =====\n")

for root, dirs, files in os.walk(PROJECT_DIR):

    for filename in files:

        path = os.path.join(root, filename)

        print(
            filename,
            "|",
            round(os.path.getsize(path) / 1024, 2),
            "KB"
        )

print("\n===== REQUIRED FILES =====")

required = [
    "train.jsonl",
    "validation.jsonl",
    "test.jsonl",
    "baseline_results.json",
    "baseline_metrics.json",
    "finetuned_results.json",
    "finetuned_metrics.json",
    "training_history.json",
    "model_comparison.csv",
    "experiment_summary.json"
]

for filename in required:

    path = os.path.join(PROJECT_DIR, filename)

    print(
        ("FOUND" if os.path.exists(path) else "MISSING"),
        filename
    )

print(
    "\nAdapter exists:",
    os.path.exists(ADAPTER_DIR)
)

===== FINAL TASK 1 FILE CHECK =====

train.jsonl | 75.2 KB
validation.jsonl | 9.41 KB
test.jsonl | 11.33 KB
baseline_results.json | 35.78 KB
baseline_metrics.json | 0.1 KB
training_history.json | 0.33 KB
finetuned_results.json | 35.09 KB
finetuned_metrics.json | 0.1 KB
model_comparison.csv | 0.21 KB
experiment_summary.json | 1.15 KB
README.md | 4.98 KB
adapter_model.safetensors | 17226.7 KB
adapter_config.json | 0.71 KB
tokenizer_config.json | 7.13 KB
special_tokens_map.json | 0.6 KB
added_tokens.json | 0.59 KB
vocab.json | 2711.75 KB
merges.txt | 1632.67 KB
tokenizer.json | 11154.29 KB

===== REQUIRED FILES =====
FOUND train.jsonl
FOUND validation.jsonl
FOUND test.jsonl
FOUND baseline_results.json
FOUND baseline_metrics.json
FOUND finetuned_results.json
FOUND finetuned_metrics.json
FOUND training_history.json
FOUND model_comparison.csv
FOUND experiment_summary.json

Adapter exists: True


In [ ]:
print("BASE MODEL METRICS")
print(json.dumps(baseline_metrics, indent=2))

print("\nFINE-TUNED MODEL METRICS")
print(json.dumps(finetuned_metrics, indent=2))

print("\nTRAINING HISTORY")
print(json.dumps(training_history, indent=2))

print("\nFINE-TUNED FAILURES:", len(failures))

print("\n7B MEMORY ESTIMATES")
print(f"FP16: {fp16_weights:.1f} GB")
print(f"Full FT: {full_ft:.1f} GB")
print(f"LoRA: {lora_low}-{lora_high} GB")
print(f"QLoRA: {qlora_low}-{qlora_high} GB")

BASE MODEL METRICS
{
  "json_validity": 100.0,
  "exact_match": 8.33,
  "intent_accuracy": 76.67,
  "field_accuracy": 52.78
}

FINE-TUNED MODEL METRICS
{
  "json_validity": 100.0,
  "exact_match": 100.0,
  "intent_accuracy": 100.0,
  "field_accuracy": 100.0
}

TRAINING HISTORY
[
  {
    "epoch": 1,
    "train_loss": 0.016035941238369558,
    "validation_loss": 0.002261194895254448
  },
  {
    "epoch": 2,
    "train_loss": 0.0009126147333881817,
    "validation_loss": 0.0016849395984676308
  },
  {
    "epoch": 3,
    "train_loss": 0.0011100932750559877,
    "validation_loss": 0.0036574219131860165
  }
]

FINE-TUNED FAILURES: 0

7B MEMORY ESTIMATES
FP16: 14.0 GB
Full FT: 112.0 GB
LoRA: 16-24 GB
QLoRA: 6-10 GB


In [ ]:
baseline_failures = [
    row for row in baseline_results
    if row["predicted"] != row["expected"]
]

print("Base model failures:", len(baseline_failures))

for i, row in enumerate(baseline_failures[:3], 1):
    print(f"\n===== BASE MODEL FAILURE {i} =====")
    print("Request:", row["request"])
    print("Expected:", json.dumps(row["expected"], indent=2))
    print("Predicted:", json.dumps(row["predicted"], indent=2))

Base model failures: 55

===== BASE MODEL FAILURE 1 =====
Request: Track my order ORD22280.
Expected: {
  "intent": "order_tracking",
  "destination": null,
  "date": null,
  "time": null,
  "party_size": null,
  "item": "ORD22280"
}
Predicted: {
  "intent": "order_tracking",
  "destination": "ORD22280",
  "date": "",
  "time": "",
  "party_size": null,
  "item": ""
}

===== BASE MODEL FAILURE 2 =====
Request: I need accommodation in Vijayawada for 3 guests.
Expected: {
  "intent": "hotel_booking",
  "destination": "Vijayawada",
  "date": null,
  "time": null,
  "party_size": 3,
  "item": null
}
Predicted: {
  "intent": "hotel_booking",
  "destination": "Vijayawada",
  "date": "today",
  "time": "",
  "party_size": 3,
  "item": ""
}

===== BASE MODEL FAILURE 3 =====
Request: Search for an air ticket to Chennai for this weekend.
Expected: {
  "intent": "flight_search",
  "destination": "Chennai",
  "date": "this weekend",
  "time": null,
  "party_size": null,
  "item": null
}
Predicted:

In [ ]:

from pathlib import Path

# Create Task 1 output folder in Google Drive
output_dir = Path("/content/drive/MyDrive/Tndrr_Task1_LoRA")
output_dir.mkdir(parents=True, exist_ok=True)

readme = """# Task 1: Intent-to-JSON Extraction Using LoRA Fine-Tuning

## 1. Objective

The objective of this task was to fine-tune a small open-weight language model to convert natural-language requests into structured JSON. The model was trained to identify the user's intent and extract relevant information into a fixed six-field schema.

## 2. Model and Dataset

- **Base model:** Qwen2.5-0.5B-Instruct
- **Fine-tuning method:** Low-Rank Adaptation (LoRA)
- **Dataset size:** 510 synthetic examples
- **Training samples:** 400
- **Validation samples:** 50
- **Testing samples:** 60
- **Random seed:** 42

The dataset covered six intents: restaurant booking, flight search, hotel booking, product search, weather query, and order tracking.

The output JSON contained six fields:

`intent`, `destination`, `date`, `time`, `party_size`, and `item`.

Fields that were not applicable or not mentioned in the request were represented using JSON `null`.

## 3. LoRA Configuration and Training

The model was fine-tuned using the PEFT library with the following configuration:

| Parameter | Value |
|---|---|
| LoRA rank (r) | 8 |
| LoRA alpha | 16 |
| LoRA dropout | 0.05 |
| Target modules | q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj |
| Learning rate | 0.0002 |
| Epochs | 3 |
| Batch size | 4 |
| Gradient accumulation | 4 |
| Optimizer | AdamW |

LoRA was selected because it updates a small number of additional trainable parameters while keeping the original model weights frozen. This reduces training memory requirements and computational cost compared with full fine-tuning.

## 4. Evaluation and Results

The base and fine-tuned models were evaluated on the same 60 test examples using JSON validity, exact-match accuracy, intent accuracy, and field-level accuracy.

| Evaluation metric | Base model | Fine-tuned model |
|---|---:|---:|
| JSON validity | 100% | 100% |
| Exact-match accuracy | 8.33% | 100% |
| Intent accuracy | 76.67% | 100% |
| Field-level accuracy | 52.78% | 100% |

The fine-tuned model achieved 100% exact-match accuracy, correctly generating all six fields for all 60 test examples. The base model produced 55 incorrect predictions, whereas the fine-tuned model produced zero incorrect predictions on this test set.

## 5. Training Loss

| Epoch | Training loss | Validation loss |
|---|---:|---:|
| 1 | 0.01604 | 0.00226 |
| 2 | 0.00091 | 0.00168 |
| 3 | 0.00111 | 0.00366 |

Training loss decreased substantially during the first two epochs. Validation loss increased during the third epoch, indicating a slight deterioration in validation performance at the end of training. However, the final model achieved 100% accuracy on the held-out test set.

## 6. Base Model Failure Analysis

Three representative failures from the base model are presented below.

**Failure 1: Incorrect tracking ID extraction**

- Input: "Track my order ORD22280."
- Expected: The tracking ID should be stored in the `item` field.
- Base model prediction: The ID was incorrectly placed in `destination`, and `item` was returned as an empty string.
- Fine-tuned model: Correctly extracted `ORD22280` into `item`.

**Failure 2: Hallucinated date**

- Input: "I need accommodation in Vijayawada for 3 guests."
- Expected: The `date` field should be null.
- Base model prediction: Generated `"today"` even though no date was specified.
- Fine-tuned model: Correctly returned null.

**Failure 3: Incorrect intent classification**

- Input: "Search for an air ticket to Chennai for this weekend."
- Expected: `flight_search`.
- Base model prediction: Generated the unsupported intent `air_ticket`.
- Fine-tuned model: Correctly predicted `flight_search`.

These examples show improvements in field extraction, handling of missing information, and adherence to the predefined intent categories.

## 7. Estimated GPU Memory Requirements for a 7B Model

The following are approximate memory estimates for a 7-billion-parameter model.

| Training approach | Estimated memory |
|---|---:|
| FP16 model weights only | 14 GB |
| Full fine-tuning | 112 GB |
| LoRA | 16–24 GB |
| QLoRA | 6–10 GB |

Full fine-tuning requires substantially more memory because it stores gradients, optimizer states, and additional training parameters. LoRA reduces this requirement by freezing the base model, while QLoRA further reduces memory by quantizing the base weights to 4-bit precision.

These figures are estimates and exclude or simplify some activation and temporary-memory requirements. Actual usage depends on sequence length, batch size, optimizer, and implementation.

## 8. Conclusion

The experiment demonstrated that LoRA fine-tuning can substantially improve structured information extraction using a small open-weight language model. The fine-tuned Qwen2.5-0.5B-Instruct model achieved 100% exact-match accuracy on the 60-example test set, compared with 8.33% for the base model.

The experiment also demonstrated the benefits of parameter-efficient fine-tuning in reducing the number of trainable parameters while preserving the original model weights.

Although the results are promising, the dataset was synthetically generated and relatively small. Further evaluation on larger, diverse, real-world requests would be necessary to establish generalization beyond this test set.
"""

readme_path = output_dir / "README.md"
readme_path.write_text(readme, encoding="utf-8")

print("Task 1 README created successfully!")
print("Saved to:", readme_path)
print("File size:", readme_path.stat().st_size, "bytes")


Task 1 README created successfully!
Saved to: /content/drive/MyDrive/Tndrr_Task1_LoRA/README.md
File size: 5295 bytes


In [ ]:
with open("/content/drive/MyDrive/Tndrr_Task1_LoRA/README.md",
          "r", encoding="utf-8") as f:
    print(f.read())

# Task 1: Intent-to-JSON Extraction Using LoRA Fine-Tuning

## 1. Objective

The objective of this task was to fine-tune a small open-weight language model to convert natural-language requests into structured JSON. The model was trained to identify the user's intent and extract relevant information into a fixed six-field schema.

## 2. Model and Dataset

- **Base model:** Qwen2.5-0.5B-Instruct
- **Fine-tuning method:** Low-Rank Adaptation (LoRA)
- **Dataset size:** 510 synthetic examples
- **Training samples:** 400
- **Validation samples:** 50
- **Testing samples:** 60
- **Random seed:** 42

The dataset covered six intents: restaurant booking, flight search, hotel booking, product search, weather query, and order tracking.

The output JSON contained six fields:

`intent`, `destination`, `date`, `time`, `party_size`, and `item`.

Fields that were not applicable or not mentioned in the request were represented using JSON `null`.

## 3. LoRA Configuration and Training

The model was fine-

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os

print("Task 1-related folders in Google Drive:\n")

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    dirs[:] = [d for d in dirs if d not in [".ipynb_checkpoints"]]

    if any(word in root.lower() for word in ["task1", "intent", "lora", "tndrr"]):
        print("\nFOLDER:", root)
        for f in files[:20]:
            print("  ", f)

Mounted at /content/drive
Task 1-related folders in Google Drive:


FOLDER: /content/drive/MyDrive/Tndrr_Task1

FOLDER: /content/drive/MyDrive/Tndrr_Task1/task1_intent2json
   train.jsonl
   test.jsonl
   validation.jsonl
   baseline_results.json
   baseline_metrics.json
   training_history.json
   finetuned_results.json
   finetuned_metrics.json
   model_comparison.csv
   experiment_summary.json

FOLDER: /content/drive/MyDrive/Tndrr_Task1/task1_intent2json/intent2json_lora_v2
   tokenizer.json
   README.md
   added_tokens.json
   adapter_config.json
   tokenizer_config.json
   special_tokens_map.json
   merges.txt
   vocab.json
   adapter_model.safetensors

FOLDER: /content/drive/MyDrive/Tndrr_Task2_RAG
   source_document.pdf
   extracted_pages.json
   retrieval_comparison.csv
   rag_experiment_results.json
   chunks_300.json
   chunks_600.json
   evaluation_questions.json
   final_task2_summary.json

FOLDER: /content/drive/MyDrive/Tndrr_Task3_Inference
   q4_results.json
   benchmark